**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**05 | Class-Weighted Temporal CNN and Matched Grouped Comparison**

![HARTH weighted-loss and grouped comparison schematic](../../images/05_consolidated_workflow.svg)

**Objective:** Compare standard and Train-derived class-weighted temporal CNNs, then benchmark the two CNN variants against Random Forest using matched participant-disjoint Development folds.

**Dataset:** Official NTNU AI Lab HARTH v2.0, with lower-back and thigh tri-axial acceleration at nominal 50 Hz. The classification task retains eight documented source activity classes (1–8).

**Protocol:** Reconstruct one shared cohort of 100-sample, label-pure windows with a 50-sample stride and a maximum of 500 windows per person. Experiment A uses one whole-participant Train/Validation/unused Development split; Experiment B uses three-fold GroupKFold over the same Development cohort. S023 and S032 are excluded throughout.

**Scope:** This is Development-only evidence. The fixed Validation and grouped cross-validation estimates are not interchangeable, and neither constitutes independent external validation.


**00. Colab Repository Setup**

Obtain the public repository checkout for the study's original source adapter, shared models and output utilities.


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

Prepare dataset, splitting, modelling, evaluation and visualisation tools. Both experiments write current stage-specific outputs into the study output directory.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# --------------- 3. Machine Learning and Deep Learning Metrics ---------------
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay,
)
import torch

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4),
)

# ------------------ 5. HARTH and Shared Research Utilities -------------------
from harth_data import (
    HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS,
    prepare_harth_window_cohort,
)
from core.data_splitting import split_group_indices
from core.temporal_cnn import train_temporal_cnn, compute_training_class_weights
from core.temporal_comparison import compare_models_group_cv
from core.ml_baselines import (
    classification_class_report, evaluate_predictions_by_subject,
)
from core.group_cross_validation import compare_models_by_subject
from core.output_download import download_outputs_from_colab

# ------------------ 6. Training Device and Output Directory ------------------
from core.computer_device import computer_device
device = computer_device()
DEVICE = device.type
output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Build the Shared Development Window Cohort**

Create one bounded, temporally continuous and label-pure source-window cohort; preserve row metadata and group IDs. Raw sequences are reconstructed from the official source rather than substituted with 30-feature statistical summaries.


In [ ]:
# ============== 02. Build the Shared Development Window Cohort ===============

# ------------------- 1. Prepare Matched Development Cohort -------------------
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))

X, y, metadata, overview = prepare_harth_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES, max_windows_per_subject=500,
    window_size=100, step_size=50, sampling_rate=50,
)

if set(metadata["subject_id"]) & RESERVED_TEST:
    raise ValueError("Reserved participants must not be loaded.")

# ------------------------ 2. Summarise Source Windows ------------------------
if not (len(X) == len(y) == len(metadata)):
    raise ValueError("Window, label and metadata rows must align.")
if X.shape[1:] != (len(SENSOR_COLUMNS), 100):
    raise ValueError("Expected six-channel, 100-sample temporal windows.")
if not np.array_equal(metadata["label"].to_numpy(), y):
    raise ValueError("Window metadata and labels are misaligned.")
print("Shared signal cohort:", X.shape)

display(overview[["subject_id", "eligible_windows", "sampled_windows"]])


**03. Inspect Class Support and Split Participants**

Summarise the eight-class support per person. For the loss-ablation experiment, create a participant-disjoint 70/20/10 Train/Validation/unused Development split with seed 42; the third partition is not the reserved Test.


In [ ]:
# ============= 03. Inspect Class Support and Split Participants ==============

# ------------------ 1. Inspect Participant Activity Support ------------------
coverage = pd.crosstab(metadata["subject_id"], metadata["label"]).reindex(
    index=DEVELOPMENT_SUBJECTS, columns=TASK_CLASSES, fill_value=0
)
coverage.columns = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]
display(coverage)

print("Independent participants supporting each class:",
      (coverage > 0).sum(axis=0).to_dict())

# ------------------ 2. Recreate Participant-Disjoint Split -------------------
parts = split_group_indices(
    metadata["subject_id"].to_numpy(),
    train_fraction=0.70, validation_fraction=0.20,
    test_fraction=0.10, random_state=42,
)
train_idx, val_idx = parts["train"], parts["validation"]

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

# --------------------- 3. Verify Participant Separation ----------------------
val_subject_ids = metadata.iloc[val_idx]["subject_id"].to_numpy()

train_subject_ids = metadata.iloc[train_idx]["subject_id"].to_numpy()
unused_subject_ids = metadata.iloc[parts["test"]]["subject_id"].to_numpy()
if set(train_subject_ids) & set(val_subject_ids) or set(unused_subject_ids) & (
    set(train_subject_ids) | set(val_subject_ids)
):
    raise ValueError("Development participant partitions overlap.")
if set(np.unique(y_train)) != set(TASK_CLASSES):
    raise ValueError("One or more target classes are absent from Train.")
print("Train windows:", len(y_train), "Validation windows:", len(y_val))
print("Unused development windows:", len(parts["test"]))


**04. Calculate Train-Only Class Weights**

Calculate inverse-square-root class-frequency weights on Train labels only and normalise their arithmetic mean to one. The labels and Validation distribution are not modified.


In [ ]:
# ================== 04. Calculate Train-Only Class Weights ===================

# --------------------- 1. Compute Training Class Weights ---------------------
training_counts = pd.Series(y_train).value_counts().reindex(TASK_CLASSES, fill_value=0)
weights = compute_training_class_weights(y_train, TASK_CLASSES, method="inverse_sqrt")  # Derive weighting solely from Train labels.

# ------------------------- 2. Inspect Class Support --------------------------
weight_table = pd.DataFrame({
    "activity": [ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    "train_windows": training_counts.to_numpy(),
    "inverse_sqrt_weight": weights,
})
if not (training_counts > 0).all():
    raise ValueError("Every defined class must occur in Train.")

display(weight_table.round(4))


**05. Train Matched Standard and Weighted CNNs**

Fit both CNN loss conditions with identical Train/Validation windows, architecture, five epochs, batch size 128, Adam learning rate 0.001 and seed 42. Only the loss weighting differs.


In [ ]:
# =============== 05. Train Matched Standard and Weighted CNNs ================

# ------------------- 1. Configure Matched CNN Experiments --------------------
settings = dict(
    X_train=X_train, y_train=y_train,
    X_validation=X_val, y_validation=y_val,
    classes=TASK_CLASSES, epochs=5, batch_size=128,
    learning_rate=1e-3, seed=42,
    device=DEVICE,
)
runs = {}

# --------------------- 2. Fit Standard and Weighted CNNs ---------------------
for method, name in [
    ("none", "Standard CNN"),
    ("inverse_sqrt", "Weighted CNN"),
]:
    print("\nTraining:", name)
    model, history, predicted, scaler = train_temporal_cnn(
        **settings, class_weighting=method  # Only the loss-weighting choice differs between runs.
    )
    runs[name] = {"model": model, "history": history, "predicted": predicted, "scaler": scaler}
    print("Last-epoch Macro F1:", float(history.iloc[-1]["validation_macro_f1"]))


**06. Compare Loss-Ablation Validation Metrics**

Calculate fixed eight-class Macro F1 and balanced accuracy and plot training/Validation curves. Cross-entropy loss values reflect different weighting objectives.


In [ ]:
# =============== 06. Compare Loss-Ablation Validation Metrics ================

# ----------------------- 1. Compare Validation Metrics -----------------------
summary = pd.DataFrame([
    {"model": name,
     "macro_f1": f1_score(
         y_val, result["predicted"], labels=TASK_CLASSES,
         average="macro", zero_division=0),
     "balanced_accuracy": balanced_accuracy_score(y_val, result["predicted"]),
     "validation_windows": len(y_val)}
    for name, result in runs.items()
])

display(summary.round(4))

# ----------------- 2. Inspect Training and Validation Curves -----------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for name, result in runs.items():
    curve = result["history"]
    axes[0].plot(curve["epoch"], curve["train_loss"], marker="o", label=name)
    axes[1].plot(curve["epoch"], curve["validation_macro_f1"], marker="o", label=name)

axes[0].set(title="Training loss (different weighting objectives)",
            xlabel="Epoch", ylabel="Weighted or standard Cross-Entropy")
axes[1].set(title="Held-out Validation Macro F1", xlabel="Epoch",
            ylabel="Macro F1", ylim=(0, 1))
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)

fig.tight_layout()
save_figure(fig, output_dir / "05_loss_training_curves.png")
plt.show()


**07. Inspect Loss-Ablation Class Errors**

Inspect per-class precision, recall, F1 and support for both CNNs, with particular attention to minority activities and matched confusion matrices.


In [ ]:
# ================== 07. Inspect Loss-Ablation Class Errors ===================

# --------------------- 1. Calculate Class-Level Metrics ----------------------
reports = []

for name, result in runs.items():
    report = classification_class_report(
        y_val, result["predicted"], classes=TASK_CLASSES,
        class_names=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    )
    report["model"] = name
    reports.append(report)
class_table = pd.concat(reports, ignore_index=True)

display(class_table[["model", "class", "support", "precision", "recall", "f1"]].round(4))

# -------------------- 2. Compare Class F1 and Confusions ---------------------
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(len(TASK_CLASSES))

for i, name in enumerate(runs):
    values = class_table.loc[class_table["model"] == name, "f1"].to_numpy()
    ax.bar(positions + (i - 0.5) * 0.36, values, width=0.34, label=name)
ax.set_xticks(positions, [ACTIVITY_NAMES[c] for c in TASK_CLASSES], rotation=40, ha="right")
ax.set(title="Per-class F1: standard vs training-weighted CNN",
       ylabel="F1 score", ylim=(0, 1))
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "05_loss_per_class_f1.png")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (name, result) in zip(axes, runs.items()):
    matrix = confusion_matrix(y_val, result["predicted"], labels=TASK_CLASSES)
    ConfusionMatrixDisplay(
        confusion_matrix=matrix, display_labels=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    ).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
    ax.set_title(name)
fig.tight_layout()
save_figure(fig, output_dir / "05_loss_confusion_matrices.png")
plt.show()


**08. Compare Paired Validation Participant Scores**

Measure fixed-task Macro F1 per Validation participant and describe weighted-minus-standard CNN differences without treating correlated windows as independent people.


In [ ]:
# ============= 08. Compare Paired Validation Participant Scores ==============

# ------------------- 1. Evaluate Paired Participant Scores -------------------
individual_results = pd.concat([
    evaluate_predictions_by_subject(
        y_val, result["predicted"], val_subject_ids,
        TASK_CLASSES, model_name=name,
    )
    for name, result in runs.items()
], ignore_index=True)
paired, paired_summary = compare_models_by_subject(
    individual_results, model_a="Weighted CNN", model_b="Standard CNN",
    score_column="macro_f1_fixed_task",
)

# ------------------- 2. Visualise Per-Participant Changes --------------------
display(paired.round(4))
display(pd.DataFrame([paired_summary]).round(4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(paired["subject_id"], paired["difference_a_minus_b"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Participant-level Macro F1: weighted minus standard",
       xlabel="Held-out participant", ylabel="Signed Macro F1 difference")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "05_loss_paired_subject_difference.png")
plt.show()


**09. Export Fixed-Split Loss-Ablation Results**

Save full-precision loss-ablation metrics, support tables, source-aligned predictions, paired participant scores, learning histories, model weights and figures in the `05_loss_` output family.


In [ ]:
# =============== 09. Export Fixed-Split Loss-Ablation Results ================

# ---------------------- 1. Save Loss-Ablation Evidence -----------------------
overview.to_csv(output_dir / "05_bounded_development_cohort_summary.csv", index=False)
pd.DataFrame([
    {"partition": "unused_development" if name == "test" else name,
     "subject_id": subject}
    for name, indices in parts.items()
    for subject in sorted(set(metadata.iloc[indices]["subject_id"]))
]).to_csv(output_dir / "05_loss_partition_membership.csv", index=False)
weight_table.to_csv(output_dir / "05_loss_training_class_weights.csv", index=False)
summary.to_csv(output_dir / "05_loss_validation_metrics.csv", index=False)
class_table.to_csv(output_dir / "05_loss_per_class_metrics.csv", index=False)
individual_results.to_csv(output_dir / "05_loss_per_subject_metrics.csv", index=False)
paired.to_csv(output_dir / "05_loss_paired_subjects.csv", index=False)
pd.DataFrame([paired_summary]).to_csv(
    output_dir / "05_loss_paired_summary.csv", index=False
)
for name, run in runs.items():
    safe_name = "weighted" if name == "Weighted CNN" else "standard"
    run["history"].to_csv(
        output_dir / f"05_loss_{safe_name}_cnn_training_history.csv", index=False
    )
    pd.DataFrame(confusion_matrix(
        y_val, run["predicted"], labels=TASK_CLASSES
    ), index=TASK_CLASSES, columns=TASK_CLASSES).rename_axis(
        "true_source_code"
    ).to_csv(output_dir / f"05_loss_{safe_name}_validation_confusion.csv")
    torch.save({
        "model_state_dict": {k: v.detach().cpu() for k, v in run["model"].state_dict().items()},
        "architecture": {"name": "TemporalCNN", "in_channels": 6,
                         "n_classes": 8, "hidden_channels": 32},
        "task_classes": list(TASK_CLASSES),
        "train_only_channel_mean": run["scaler"]["channel_mean"].tolist(),
        "train_only_channel_std": run["scaler"]["channel_std"].tolist(),
        "class_weighting": "inverse_sqrt" if safe_name == "weighted" else "none",
    }, output_dir / f"05_loss_{safe_name}_cnn.pt")
pd.DataFrame({
    "source_row": np.tile(val_idx, len(runs)),
    "subject_id": np.tile(val_subject_ids, len(runs)),
    "true_source_code": np.tile(y_val, len(runs)),
    "model": np.repeat(list(runs.keys()), len(y_val)),
    "predicted_source_code": np.concatenate([r["predicted"] for r in runs.values()]),
}).to_csv(output_dir / "05_loss_validation_predictions.csv", index=False)
print("Saved Development Validation loss-ablation evidence in:", output_dir)


**10. Run Three-Model Matched GroupKFold**

Evaluate Random Forest, standard CNN and weighted CNN in three whole-participant folds. Random Forest uses 30 deterministic features and 120 trees; CNNs use raw six-channel windows and Train-only scaling. Each training fold determines its own loss weights.


In [ ]:
# ================== 10. Run Three-Model Matched GroupKFold ===================

# -------------------- 1. Set Grouped Comparison Protocol ---------------------
FOLDS = 3
EPOCHS = 5


# --------------------- 2. Run Matched GroupKFold Models ----------------------
fold_results, subject_results, class_results, prediction_records = compare_models_group_cv(
    windows=X,
    labels=y,
    subject_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    channel_names=SENSOR_COLUMNS,
    n_splits=FOLDS,
    epochs=EPOCHS,
    batch_size=128,
    learning_rate=1e-3,
    random_state=42,
    device=DEVICE,
    forbidden_subject_ids=RESERVED_TEST,  # Exclude reserved Test participants from the evaluation.
)

# --------------------- 3. Inspect Held-Out Fold Results ----------------------
display(fold_results[[
    "fold", "model", "validation_subjects", "train_windows",
    "validation_windows", "validation_classes_present",
    "macro_f1", "balanced_accuracy"
]].round(4))


**11. Compare Grouped Fold-Level Scores**

Report three-fold Macro F1 means and fold standard deviations and Balanced Accuracy. Fold-level statistics are descriptive because training folds share participants.


In [ ]:
# =================== 11. Compare Grouped Fold-Level Scores ===================

# -------------------- 1. Summarise Fold-Level Performance --------------------
fold_summary = fold_results.groupby("model", as_index=False).agg(
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_sd=("macro_f1", "std"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
)

display(fold_summary.round(4))

# ------------------------ 2. Visualise Fold Variation ------------------------
fig, ax = plt.subplots(figsize=(10, 5))

for name, table in fold_results.groupby("model", sort=False):
    table = table.sort_values("fold")
    ax.plot(table["fold"], table["macro_f1"], marker="o", label=name)

ax.set(title="HARTH: matched development GroupKFold Macro F1",
       xlabel="Held-out fold", ylabel="Macro F1", ylim=(0, 1),
       xticks=list(range(1, FOLDS + 1)))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
save_figure(fig, output_dir / "05_cv_fold_macro_f1.png")
plt.show()


**12. Inspect Grouped Out-of-Fold Activity F1**

Compare class-level metrics on the matched out-of-fold Development windows; every source row contributes once per model.


In [ ]:
# ================ 12. Inspect Grouped Out-of-Fold Activity F1 ================

# --------------------- 1. Calculate Pooled Class Metrics ---------------------
class_results["activity"] = class_results["class"].map(
    lambda code: ACTIVITY_NAMES[int(code)]
)

display(class_results[[
    "model", "activity", "support", "precision", "recall", "f1"
]].round(4))
fig, ax = plt.subplots(figsize=(13, 5))

# ------------------------- 2. Compare Matched Models -------------------------
names = ["Random Forest", "Standard CNN", "Weighted CNN"]
positions = np.arange(len(TASK_CLASSES))
bar_width = 0.25

for i, name in enumerate(names):
    values = class_results.loc[class_results["model"] == name, "f1"].to_numpy()
    ax.bar(positions + (i-1)*bar_width, values, width=bar_width, label=name)
ax.set_xticks(positions, [ACTIVITY_NAMES[c] for c in TASK_CLASSES],
              rotation=35, ha="right")

ax.set(title="Out-of-fold activity F1 by model",
       ylabel="Pooled F1", ylim=(0, 1))
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "05_cv_per_class_f1.png")
plt.show()


**13. Compare Paired Development Participant Effects**

Compare weighted CNN with standard CNN and Random Forest using the same 29 participants and fixed eight-class Macro F1. Preserve both signed differences.


In [ ]:
# ============ 13. Compare Paired Development Participant Effects =============

# ------------------- 1. Calculate Participant-Level Scores -------------------
display(subject_results[[
    "fold", "model", "subject_id", "windows",
    "observed_classes", "macro_f1_fixed_task"
]].sort_values(["subject_id", "model"]).round(4))
comparisons = [
    ("Weighted CNN", "Standard CNN"),
    ("Weighted CNN", "Random Forest"),
]

# ---------------------- 2. Compare Paired Participants -----------------------
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

for ax, (model_a, model_b) in zip(axes, comparisons):
    paired, pair_summary = compare_models_by_subject(
        subject_results, model_a=model_a, model_b=model_b,
        score_column="macro_f1_fixed_task",
    )
    display(pd.DataFrame([pair_summary]).round(4))
    ax.bar(paired["subject_id"], paired["difference_a_minus_b"])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=f"{model_a} minus {model_b}",
           ylabel="Paired fixed-task Macro F1")
    ax.grid(axis="y", alpha=0.2)

axes[-1].tick_params(axis="x", rotation=80)
axes[-1].set_xlabel("Held-out development participant")
fig.tight_layout()
save_figure(fig, output_dir / "05_cv_paired_subject_f1.png")
plt.show()


**14. Export Matched GroupKFold Evidence**

Save three-model fold, class and participant metrics, paired comparisons and complete out-of-fold predictions under `05_cv_` names, then package the stage's current research outputs.


In [ ]:
# ================== 14. Export Matched GroupKFold Evidence ===================

# ------------------------- 1. Protect OOF Alignment --------------------------
if set(subject_results["subject_id"]) != set(DEVELOPMENT_SUBJECTS):
    raise ValueError("Out-of-fold evaluation is missing Development participants.")
if set(subject_results["subject_id"]) & RESERVED_TEST:
    raise ValueError("Reserved participants appeared in Development evaluation.")
if len(fold_results) != 3 * FOLDS:
    raise ValueError("Some model/fold evaluations are missing.")
for model_name, frame in prediction_records.groupby("model"):
    ordered = frame.sort_values("source_row")
    if ordered["source_row"].tolist() != list(range(len(X))):
        raise ValueError(f"Missing or duplicated OOF source rows: {model_name}.")
    if not np.array_equal(ordered["true_label"].to_numpy(), y):
        raise ValueError(f"Out-of-fold source labels are misaligned: {model_name}.")

# ------------------------ 2. Save Grouped CV Evidence ------------------------
overview.to_csv(output_dir / "05_cv_development_cohort_summary.csv", index=False)
coverage.rename_axis("subject_id").reset_index().to_csv(
    output_dir / "05_cv_development_activity_coverage.csv", index=False
)
fold_results.to_csv(output_dir / "05_cv_fold_metrics.csv", index=False)
fold_summary.to_csv(output_dir / "05_cv_model_summary.csv", index=False)
subject_results.to_csv(output_dir / "05_cv_subject_metrics.csv", index=False)
class_results.to_csv(output_dir / "05_cv_per_class_metrics.csv", index=False)
prediction_records.to_csv(output_dir / "05_cv_oof_predictions.csv", index=False)

paired_rows, pair_summaries = [], []
for model_a, model_b in comparisons:
    pair_table, pair_stats = compare_models_by_subject(
        subject_results, model_a=model_a, model_b=model_b,
        score_column="macro_f1_fixed_task",
    )
    pair_table = pair_table.copy()
    pair_table["model_a"] = model_a
    pair_table["model_b"] = model_b
    paired_rows.append(pair_table)
    pair_summaries.append({"model_a": model_a, "model_b": model_b, **pair_stats})
pd.concat(paired_rows, ignore_index=True).to_csv(
    output_dir / "05_cv_paired_subjects.csv", index=False
)
pd.DataFrame(pair_summaries).to_csv(
    output_dir / "05_cv_paired_summary.csv", index=False
)
print("Saved Development-only matched model evidence in:", output_dir)

download_outputs_from_colab(folder="wearable_activity_recognition/output", stage="05")


**15. Scientific Interpretation and Research Boundaries**

**Dataset and independent unit:** The bounded eight-class HARTH cohort contains temporally contiguous, label-pure two-second windows. A 500-window per-person cap changes the activity distribution; overlapping windows cannot be treated as independent participants.

**Loss-ablation question:** The two CNNs differ only in Train-derived cross-entropy weighting on the same Train/Validation groups. Class-specific and participant-paired results complement aggregate Macro F1.

**Grouped-comparison question:** Three participant-disjoint folds compare configured Random Forest, standard CNN and weighted CNN pipelines on matched held-out windows. Different CNN raw-signal and Random Forest statistical representations prevent attribution of observed differences solely to architecture.

**Scope:** Fixed Validation and grouped Development evaluations use different protocols and are not pooled. Neither establishes independent external generalisation, real-time performance or statistical superiority; S023 and S032 are not assessed here.
